# Import Dependencies

In [1]:
import pandas as pd
import plotly.express as px
import logging
from pathlib import Path
from tools import transformer
from scripts import downloader

/home/hwangwy/Projects/Python/Assignments/Ecommerce-Analyst/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Ingest Dataset

In [3]:
raw_df_ls = {}

try:
    url = "erfan4524/e-commerce-sales-data-analysis-and-eda"
    data_path = 'data/'
    downloader.dataset_download(url, output_dir=data_path)

    for file in Path(data_path).iterdir():
        file_name = Path(file).name
        if file_name == ".complete" or file_name == "clean_final_data.csv":
            continue
        else:
            logger.info(f"CSV file founded: {file_name}")
            raw_df_ls[file_name] = pd.read_csv(f"{data_path}/{file_name}")
            logger.info(f"Added into df_ls")
except Exception as e:
    logger.error(f"Exception occurred. Detail: {e}")

2026-09-15 15:31:57,713 - scripts.downloader - INFO - Dataset is already downloaded at data/
2026-09-15 15:31:57,713 - __main__ - INFO - CSV file founded: customers.csv
2026-09-15 15:31:57,722 - __main__ - INFO - Added into df_ls
2026-09-15 15:31:57,722 - __main__ - INFO - CSV file founded: orders.csv
2026-09-15 15:31:57,753 - __main__ - INFO - Added into df_ls
2026-09-15 15:31:57,753 - __main__ - INFO - CSV file founded: payments.csv
2026-09-15 15:31:57,773 - __main__ - INFO - Added into df_ls
2026-09-15 15:31:57,773 - __main__ - INFO - CSV file founded: products.csv
2026-09-15 15:31:57,774 - __main__ - INFO - Added into df_ls


# Dataset Information

These block of codes will show the data inside csv files and show what's need to clean

# Customers.csv

In [4]:
df_ls: dict[str, pd.DataFrame] = {}

In [5]:
raw_df: pd.DataFrame = raw_df_ls['customers.csv']
# transformer.view_data(raw_df)
print(raw_df[(raw_df['Age'].isna() & raw_df['City'].isna())])
print(raw_df['CustomerSegment'].unique())

      CustomerID  Age City  SignupDate CustomerSegment
2794      102795  NaN  NaN  2023-12-16             New
3633      103634  NaN  NaN  2025-08-16             New
8532      108533  NaN  NaN  2023-01-14             New
8677      108678  NaN  NaN  2025-07-28             New
<ArrowStringArray>
['Regular', 'VIP', 'New']
Length: 3, dtype: str


First look into the dataset, we will se Age column is float64 instead of int64, SignupDate is str instead of date. Secondly is handle null values, in the Age column instead of dropping Age, we will fill it with mean values and the change the datatype into int64. For City column because it contains 12 unique values so we drop it all (Note that these two column have a total of 1000 rows so you can't drop 300+ rows of both values. This will affect the quality of the dataset)

Run the pipeline in the cleaner tools

In [6]:
df = transformer.clean_customer(raw_df)
# transformer.view_data(df)
print(df[df['City'].isna()])
df_ls['customers.csv'] = df

2026-09-15 15:31:57,822 - tools.transformer - INFO - Columns formatted
2026-09-15 15:31:57,825 - tools.transformer - INFO - Dropped duplicates
2026-09-15 15:31:57,826 - tools.transformer - INFO - Filled Age Null values to mean
2026-09-15 15:31:57,826 - tools.transformer - INFO - Changed Age dtype to int
2026-09-15 15:31:57,829 - tools.transformer - INFO - Dropped Null values from City
2026-09-15 15:31:57,832 - tools.transformer - INFO - Formatted SignupDate to type datetime


Empty DataFrame
Columns: [CustomerID, Age, City, SignupDate, CustomerSegment]
Index: []


# Orders.csv

In [7]:
df: pd.DataFrame = raw_df_ls['orders.csv']
# transformer.view_data(df)
print(df[df['Quantity'].isna()].head(5))
print(df['PaymentMethod'].unique())
print(df[df['PaymentMethod'].isna()].head(5))
print(df['Status'].unique())

      OrderID  CustomerID   OrderDate  ProductID  Quantity  Discount  \
1725   501726      103832  2024-03-17       2006       NaN      0.05   
2664   502665      106550  2025-06-24       2003       NaN      0.10   
3521   503522      105703  2024-05-08       2005       NaN      0.00   
4801   504802      104924  2024-08-18       2003       NaN      0.30   
5312   505313      107357  2024-02-07       2019       NaN      0.05   

     PaymentMethod     Status  
1725       Gateway  Completed  
2664    CardToCard  Completed  
3521        Wallet  Completed  
4801        Wallet  Completed  
5312       Gateway  Completed  
<ArrowStringArray>
['Gateway', 'Wallet', 'CardToCard', 'Cash', nan]
Length: 5, dtype: str
     OrderID  CustomerID   OrderDate  ProductID  Quantity  Discount  \
44    500045      100241  2025-06-18       2017       2.0      0.05   
113   500114      107642  2025-11-30       2007       3.0      0.00   
115   500116      102851  2026-06-04       2001       1.0      0.05   
1

In the orders.csv file, we will see that the OrderDate have the same issue with customers.csv which datatype is string instead of date, Quantity is float64 and have some outlier values that < 0. Discount is 0.x format which is hard to read. In this file, there are four columns with null values, we should drop OrderDate because dropping it won't affect much the dataset which has 50k of rows dropping 35 rows is not a big deal. For quantity columns, if we fill these values with a single value or mean it will go wrong. We should drop these values instead of filling it. The discount can be filled with 0 value instead of dropping it. Lastly, PaymentMethod can just be filled with Cash

In [8]:
df = transformer.clean_orders(df)
# transformer.view_data(df)
df_ls['orders.csv'] = df

2026-09-15 15:31:57,886 - tools.transformer - INFO - Columns formatted
2026-09-15 15:31:57,897 - tools.transformer - INFO - Dropped duplicates
2026-09-15 15:31:57,902 - tools.transformer - INFO - Dropped OrderDate Null values
2026-09-15 15:31:57,912 - tools.transformer - INFO - Formatted OrderDate dtype to date
2026-09-15 15:31:57,913 - tools.transformer - INFO - Filled Discount Null values with 0
2026-09-15 15:31:57,919 - tools.transformer - INFO - Changed dtype from float to int for Quantity column
2026-09-15 15:31:57,923 - tools.transformer - INFO - Dropped outlier values in Quantity column
2026-09-15 15:31:57,926 - tools.transformer - INFO - Filled Null value from PaymentMethod to Cash


Results after running the pipeline

# Payments.csv

In [9]:
df = raw_df_ls["payments.csv"]
# transformer.view_data(df)

This dataset has the same problems that PaymentDate is str and contains null. Because it only has 35 null values so we will drop them.

In [10]:
df = transformer.clean_payments(df)
# transformer.view_data(df)
df_ls['payments.csv'] = df

2026-09-15 15:31:57,985 - tools.transformer - INFO - Columns formatted
2026-09-15 15:31:57,994 - tools.transformer - INFO - Dropped duplicates
2026-09-15 15:31:57,997 - tools.transformer - INFO - Dropped Null values from PaymentDate
2026-09-15 15:31:58,008 - tools.transformer - INFO - Changed PaymentDate to type date


In [11]:
df = raw_df_ls['products.csv']
# transformer.view_data(df)
df_ls['products.csv'] = df

This file does not contain anything to clean.

Final dataframe for analytics

In [12]:
df = transformer.to_analytics(df_ls)
transformer.view_data(df)

2026-09-15 15:31:58,078 - tools.transformer - INFO - Starting transform cleaned data to ready to use data for analytics


===========HEAD==============
   CustomerID  Age    City CustomerSegment  OrderID  OrderDate  ProductID  \
0      100001   22  Tehran         Regular   508055 2026-02-06       2008   
1      100001   22  Tehran         Regular   512288 2025-01-01       2013   
2      100001   22  Tehran         Regular   512819 2025-05-19       2014   
3      100001   22  Tehran         Regular   528733 2026-05-29       2018   
4      100001   22  Tehran         Regular   537516 2024-05-30       2010   

   Quantity  Discount PaymentMethod     Status   ProductName     Category  \
0         4      0.00    CardToCard  Completed    Phone Case  Accessories   
1         1      0.15    CardToCard  Completed      Notebook   Stationery   
2         4      0.10        Wallet  Completed      Backpack  Accessories   
3         3      0.10       Gateway  Completed    Microphone  Electronics   
4         5      0.00        Wallet  Completed  Fitness Band    Wearables   

   UnitPrice  PaymentID PaymentStatus  
0   

# Load Phase

Data visualization using plotly express and plotly graph objects

# Company Total Sales And Revenue Over Years, Months.
This section will visualize the company sales and revenue over 2024, 2025, 2026 and every month in each year using bar char, line chart.

Total sales and revenue over year using bar charts. Used to compare sales and profit between each year.

In [37]:
total_sales = (
    df[df['Status'] == 'Completed']
    .groupby(df['OrderDate'].dt.year.rename('Year'))['OrderID']
    .size()
    .reset_index(name='TotalSales')
)

In [38]:
fig = px.bar(
    total_sales,
    width=500,
    height=600,
    x='Year',
    y='TotalSales',
    title='Total Sale by Year',
    text_auto=True
)
fig.show()

In [35]:
df['Revenue'] = (df['UnitPrice'] - df['UnitPrice'] * df['Discount']) * df['Quantity']

total_revenue = (
    df[df['Status'] == 'Completed']
    .groupby(df['OrderDate'].dt.year.rename('Year'))['Revenue']
    .sum()
    .reset_index(name='TotalRevenue')
)

In [36]:
fig = px.bar(
    total_revenue,
    width=500,
    height=600,
    x='Year',
    y='TotalRevenue',
    title='Total Revenue Over Years',
    text_auto=True
)
fig.show()

In [39]:
total_sales = (
    df[df['Status'] == 'Completed']
    .groupby([df['OrderDate'].dt.year.rename('Year'), df['OrderDate'].dt.month.rename('Month')])
    .size()
    .reset_index(name='TotalSales')
)

In [40]:
fig = px.line(
    total_sales,
    x='Month',
    y='TotalSales',
    title='Total Sales Over Month of 2024, 2025, 2026',
    color='Year',
    line_shape='spline'
)
fig.show()

We can also count the number of customers segment for every year. Eg: How many new customers do we have in 2025?. Which is very helpful for decreasing customers churn.

In [31]:
customer_segment_count = (
    df.groupby([df['OrderDate'].dt.year.rename('Year'), 'CustomerSegment'])
    .size()
    .reset_index(name='Total')
)

In [32]:
fig = px.bar(
    customer_segment_count,
    width=500,
    height=600,
    x='Year',
    y='Total',
    title='Customer Segment Count',
    color='CustomerSegment',
    text_auto=True
)
fig.show()

Products sale over year using pie chart to see which one we need to focus on. We only show 5 most sale products every year, other products will be combined as 'others'.

In [33]:
years = df['OrderDate'].dt.year.unique()
products_sales_ls: dict[int, pd.DataFrame] = {}
for year in years:
    products_sales = (
        df[(df['Status'] == 'Completed') & (df['OrderDate'].dt.year == year)]
        .groupby('ProductName')
        .size()
        .reset_index(name='TotalSales')
    )
    products_sales = products_sales.sort_values(by='TotalSales', ascending=False)
    others_total = products_sales['TotalSales'].iloc[5:].sum()
    products_sales = products_sales.iloc[:5]
    products_sales = pd.concat([products_sales, pd.DataFrame({'ProductName' : ['Others'], 'TotalSales' : [others_total]})], ignore_index=True)
    products_sales_ls[year] = products_sales

In [34]:
for year in products_sales_ls:
    fig = px.pie(
        products_sales_ls[year],
        width=600,
        height=600,
        names='ProductName',
        values='TotalSales',
        title=f'Products sale percentage in {year}',
        hole=0.4,
    )
    fig.update_traces(
        textposition='inside',
        textinfo='label+percent'
    )
    # fig.show()